# Practical Work 4 - Training a Classifier on Google Colab with PyTorch

CIFAR-10 classifier with PyTorch, run on CPU, GPU and TPU to compare the computation times.

How to use: **Runtime > Change runtime type**, choose CPU, GPU (T4) or TPU, then **Runtime > Run all**.
The last cell prints the times for the chosen hardware.

## 0. Choose the device (CPU, GPU or TPU)

In [ ]:
import torch

# PyTorch does not use the TPU by itself, it needs the torch_xla library
try:
    import torch_xla.core.xla_model as xm
    USE_TPU = True
except ImportError:
    USE_TPU = False

if USE_TPU:
    device = xm.xla_device()
elif torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")

print("Device:", device)


def sync():
    # GPU and TPU work asynchronously: wait for them before reading the time
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif USE_TPU:
        xm.wait_device_ops()

## 1. Download the CIFAR10 dataset

In [ ]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# Download and define dataloader
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
# Hardware of this runtime
!nvidia-smi || echo "No NVIDIA GPU"
!lscpu | grep "Model name"

## 2. Show some samples

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Functions to show an image
def imshow(img):
    img = img / 2 + 0.5     # Unnormalize
    npimg = img.cpu().numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Show images
imshow(torchvision.utils.make_grid(images))
# Print labels
print(' '.join('%5s' % classes[labels[j]] for j in range(4)))

## 3. Define the CNN model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()
print(net)

## 4. Define some Hyperparameters

In [ ]:
import torch.optim as optim

# move the model to the device before creating the optimizer
net.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

## 5. Training stage

In [ ]:
import time

sync()
start = time.time()

for epoch in range(2):  # Loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # Get the inputs
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # Forward + Backward + Optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        if USE_TPU:
            xm.mark_step()  # run the step on the TPU

        # Print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 2000))
            running_loss = 0.0

sync()
end = time.time()
training_time = end - start

print('Finished Training')
print(f"Training time: {training_time:.1f} sec")

## 6. Testing stage

In [ ]:
############################
#Error number 1: Input type#
############################
# fixed: the test images were not sent to the device
# (the code moved "inputs" from the training loop instead of "images")
dataiter = iter(testloader)
images, labels = next(dataiter)

# Print images
imshow(torchvision.utils.make_grid(images))
print('GroundTruth: ', ' '.join('%5s' % classes[labels[j]] for j in range(4)))

images, labels = images.to(device), labels.to(device)
outputs = net(images)

_, predicted = torch.max(outputs, 1)

print('Predicted: ', ' '.join('%5s' % classes[predicted[j]]
                              for j in range(4)))

## 7. Global performance (inference time)

In [ ]:
############################
#Error number 2: Input type#
############################
# fixed: images and labels are sent to the device
correct = 0
total = 0

sync()
start = time.time()

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

sync()
inference_time = time.time() - start
accuracy = 100 * correct / total

print('Accuracy of the network on the 10000 test images: %d %%' % accuracy)
print(f"Inference time: {inference_time:.1f} sec")

## 8. Precisions per class

In [ ]:
############################
#Error number 3: Input type#
############################
# fixed: images and labels are sent to the device
class_correct = list(0. for i in range(10))
class_total = list(0. for i in range(10))
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        c = (predicted == labels).squeeze()
        for i in range(4):
            label = labels[i].item()
            class_correct[label] += c[i].item()
            class_total[label] += 1

for i in range(10):
    print('Accuracy of %5s : %2d %%' % (
        classes[i], 100 * class_correct[i] / class_total[i]))

## 9. Summary for this runtime

In [ ]:
hardware = "TPU" if USE_TPU else ("GPU" if device.type == "cuda" else "CPU")
print(f"Hardware       : {hardware} ({device})")
print(f"Training time  : {training_time:.1f} sec (2 epochs, 12,500 batches of 4 images each)")
print(f"Inference time : {inference_time:.1f} sec (10,000 test images)")
print(f"Accuracy       : {accuracy:.1f} %")